# Лекция 6. Эмбеддинги, поиск, перенос и адаптация моделей — код

Код из лекции: метрическое обучение (парная и тройная потери, InfoNCE, температура, ArcFace, диагностика пространства эмбеддингов), поиск похожих объектов (точный поиск, LSH, IVF, PQ, nDCG), перенос обучения (заморозка, частичное дообучение, разные скорости, L2-SP, LoRA), адаптация к новому домену (AdaBN, псевдометки, CORAL, DANN) и многозадачное обучение (балансировка потерь, PCGrad).

Сквозной пример — **двузначные числа**: картинки 8×16 из двух цифр 8×8, 100 классов. На 70 числах обучаем, 30 других модель впервые видит при проверке. Глава 4 работает с одиночными цифрами и «другим сканером».

Ячейки выполняются **по порядку**: каждая следующая использует переменные из предыдущих. Проще всего — «Среда выполнения → Выполнить все». Видеокарта не нужна: весь ноутбук выполняется на процессоре примерно за минуту. Время в выводе и последние знаки чисел у вас могут немного отличаться.

На своём компьютере понадобятся `pip install torch scikit-learn`.

## Глава 1. Метрическое обучение

### 1.1. Зачем учить меру похожести

Собираем данные и инструменты. Картинки цифр делим на две части: из 60% собираем обучающие числа, из 40% — проверочные. `X_train` — 70 «знакомых» чисел по 40 картинок, `X_seen` — те же числа из проверочных картинок, `X_new` — 30 новых чисел.

* `Encoder` — кодировщик: три линейных слоя, на выходе вектор из 64 чисел, поделённый на свою длину.
* `recall_at_k` — главная метрика **Recall@k**: каждый пример по очереди становится запросом, и мы проверяем, есть ли среди `k` самых похожих на него пример того же класса. Recall@1 — точность «по ближайшему соседу».
* `report` печатает Recall@1 на знакомых числах и Recall@1 и Recall@5 на новых. Первым проверяем поиск прямо по пикселям.

In [ ]:
import copy
import time
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.set_num_threads(4)                              # одинаковая скорость на разных машинах

# картинки цифр 8×8: 60% — на сборку обучающих чисел, 40% — на сборку проверочных
X_digits, y_digits = load_digits(return_X_y=True)
X_digits = torch.tensor(X_digits, dtype=torch.float32).view(-1, 8, 8) / 16   # яркость от 0 до 1
y_digits = torch.tensor(y_digits)
Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    X_digits, y_digits, test_size=0.4, random_state=42, stratify=y_digits)

def make_numbers(X, y, numbers, seed):
    """Картинки 8×16 двузначных чисел: слева цифра десятков, справа — единиц."""
    g = torch.Generator().manual_seed(seed)
    by_digit = [torch.nonzero(y == d).flatten() for d in range(10)]   # номера картинок каждой цифры
    def pick(digits):
        """Для каждой цифры из digits — номер случайной картинки этой цифры."""
        idx = torch.empty_like(digits)
        for d in range(10):
            mask = digits == d
            idx[mask] = by_digit[d][torch.randint(len(by_digit[d]), (int(mask.sum()),), generator=g)]
        return idx
    numbers = torch.as_tensor(numbers)
    images = torch.cat([X[pick(numbers // 10)], X[pick(numbers % 10)]], dim=2)   # две 8×8 рядом → 8×16
    return images.view(len(numbers), -1), numbers     # 128 пикселей в строку и само число

# 70 чисел видим при обучении, 30 — только при проверке
perm = torch.randperm(100, generator=torch.Generator().manual_seed(0))
seen, new = perm[:70], perm[70:]
X_train, num_train = make_numbers(Xd_train, yd_train, seen.repeat_interleave(40), seed=1)   # по 40 картинок
X_seen, num_seen = make_numbers(Xd_test, yd_test, seen.repeat_interleave(20), seed=2)       # по 20
X_new, num_new = make_numbers(Xd_test, yd_test, new.repeat_interleave(20), seed=3)          # по 20

# стандартизация по обучающим картинкам; 0.01 — чтобы не делить на ноль у пустых краёв
mu, sd = X_train.mean(0), X_train.std(0) + 0.01
X_train, X_seen, X_new = (X_train - mu) / sd, (X_seen - mu) / sd, (X_new - mu) / sd
# номер класса 0–69 для функций потерь, которым нужен номер, а не само число
y_train = torch.searchsorted(seen.sort().values, num_train)

class Encoder(nn.Module):
    """Кодировщик: картинка 8×16 → эмбеддинг длины 64 на единичной сфере."""
    def __init__(self, d_in=128, width=256, d_emb=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, width), nn.ReLU(),
                                 nn.Linear(width, width), nn.ReLU(),
                                 nn.Linear(width, d_emb))

    def forward(self, x):
        return F.normalize(self.net(x), dim=1)        # делим на длину: |e| = 1

def recall_at_k(emb, labels, k=1):
    """Доля запросов, у которых среди k ближайших (кроме самого запроса) есть пример того же класса."""
    emb = F.normalize(emb, dim=1)
    sim = emb @ emb.T                                 # косинусные сходства всех пар
    sim.fill_diagonal_(-float("inf"))                 # сам себя не ищем
    nearest = sim.topk(k, dim=1).indices              # номера k самых похожих
    return (labels[nearest] == labels[:, None]).any(dim=1).float().mean().item()

def report(name, encode):
    """Recall@1 на знакомых числах, Recall@1 и Recall@5 на новых."""
    with torch.no_grad():
        e_seen, e_new = encode(X_seen), encode(X_new)
    print(f"{name:12s} знакомые R@1 {recall_at_k(e_seen, num_seen):.3f} | "
          f"новые R@1 {recall_at_k(e_new, num_new):.3f}, R@5 {recall_at_k(e_new, num_new, 5):.3f}")

print("обучение:", tuple(X_train.shape), "| знакомые:", tuple(X_seen.shape), "| новые:", tuple(X_new.shape))
print("новые числа:", sorted(new.tolist()))
report("пиксели", lambda x: x)                        # поиск прямо по пикселям, без обучения

**Что получилось.** Поиск по пикселям работает плохо: Recall@1 на знакомых числах 0.669 — у трети запросов ближайшая картинка изображает другое число.

### 1.2. Расстояние и сходство

Для векторов длины 1 квадрат расстояния и косинус связаны точной формулой $\lVert \hat{x} - \hat{y} \rVert^2 = 2(1 - \cos\theta)$. Проверим её на пяти парах случайных векторов.

In [ ]:
torch.manual_seed(0)
a, b = F.normalize(torch.randn(5, 64), dim=1), F.normalize(torch.randn(5, 64), dim=1)   # 5 пар векторов длины 1
d2 = ((a - b) ** 2).sum(dim=1)                        # квадрат евклидова расстояния
cos = (a * b).sum(dim=1)                              # скалярное произведение = косинус для векторов длины 1
print("d²         :", [round(v, 4) for v in d2.tolist()])
print("2·(1 − cos):", [round(v, 4) for v in (2 * (1 - cos)).tolist()])

**Что получилось.** Числа совпали. Квадрат расстояния около 2, то есть косинус около 0: случайные направления в пространстве из 64 чисел почти перпендикулярны.

### 1.3. Парная и тройная функции потерь

Парная потеря (contrastive loss) и тройная потеря (triplet loss) сразу для всех пар и троек батча, а также доли лёгких, полутрудных и трудных троек. Считаем всё на случайном батче и **необученном** кодировщике.

In [ ]:
def contrastive_loss(e, labels, margin=0.5):
    """Парная потеря по всем парам батча: похожие тянем друг к другу, непохожие — расталкиваем до margin."""
    d = torch.cdist(e, e)                             # расстояния между всеми парами
    same = (labels[:, None] == labels[None, :]).float()   # s = 1 для пар одного класса
    off = 1 - torch.eye(len(e))                       # пару «пример сам с собой» не считаем
    loss = same * d ** 2 + (1 - same) * F.relu(margin - d) ** 2
    return (loss * off).sum() / off.sum()

def triplet_loss(e, labels, margin=0.2):
    """Тройная потеря с поиском трудных примеров в батче (batch hard)."""
    d = torch.cdist(e, e)
    same = labels[:, None] == labels[None, :]
    hardest_pos = (d * same).max(dim=1).values        # самый далёкий позитив для каждого якоря
    hardest_neg = (d + same * 1e9).min(dim=1).values  # самый близкий негатив (своих исключаем)
    return F.relu(hardest_pos - hardest_neg + margin).mean()

def triplet_types(e, labels, margin=0.2):
    """Доли лёгких, полутрудных и трудных среди всех троек (якорь, позитив, негатив) батча."""
    d = torch.cdist(e, e)
    same = labels[:, None] == labels[None, :]
    pos = same & ~torch.eye(len(e), dtype=torch.bool)
    d_ap = d[:, :, None]                              # d(a, p): ось 1 — позитив
    d_an = d[:, None, :]                              # d(a, n): ось 2 — негатив
    valid = pos[:, :, None] & ~same[:, None, :]       # тройки «свой позитив, чужой негатив»
    easy = (d_an > d_ap + margin) & valid
    hard = (d_an < d_ap) & valid
    semi = valid & ~easy & ~hard
    n = valid.sum()
    return {k: round((v.sum() / n).item(), 2) for k, v in [("лёгкие", easy), ("полутрудные", semi), ("трудные", hard)]}

torch.manual_seed(0)
idx = torch.randperm(len(X_train))[:64]
with torch.no_grad():
    e = Encoder()(X_train[idx])                       # необученный кодировщик
print("парная потеря:", round(contrastive_loss(e, y_train[idx]).item(), 3),
      "| тройная потеря:", round(triplet_loss(e, y_train[idx]).item(), 3))
print("тройки в случайном батче:", triplet_types(e, y_train[idx]))

* Парная потеря почти нулевая ещё до обучения: позитивных пар в случайном батче мало, а негативные и так дальше отступа.
* Тройная потеря около 0.2 — около отступа; треть троек лёгкие.

### 1.4. Из чего собрать батч

Батчи **m-per-class**: 16 случайных классов по 4 примера, чтобы у каждого примера были позитивы. Для сравнения — сколько классов в обычном случайном батче представлены единственным примером.

In [ ]:
def class_batches(P=16, m=4, steps=1500, seed=0):
    """Батчи из P классов по m примеров: в каждом гарантированно есть позитивы."""
    g = torch.Generator().manual_seed(seed)
    by_class = [torch.nonzero(y_train == c).flatten() for c in range(70)]
    for _ in range(steps):
        classes = torch.randperm(70, generator=g)[:P]          # P случайных классов
        idx = torch.cat([by_class[c][torch.randperm(len(by_class[c]), generator=g)[:m]] for c in classes])
        yield X_train[idx], y_train[idx]

xb, yb = next(class_batches())
print("размер батча:", len(yb), "| классов:", len(yb.unique()), "| примеров каждого класса:", yb.bincount().max().item())
idx = torch.randperm(len(X_train), generator=torch.Generator().manual_seed(0))[:64]
counts = y_train[idx].bincount(minlength=70)
print("случайный батч из 64: классов без пары —", int((counts == 1).sum()), "из", int((counts > 0).sum()))

**Что получилось.** В случайном батче 42 класса, и у 26 из них один пример: 40% батча годятся только в негативы. В батче m-per-class таких нет.

### 1.5. Обучаем кодировщики

Обучаем одинаковые кодировщики с четырьмя функциями потерь: кросс-энтропия классификатора (для сравнения), парная, тройная и **InfoNCE** — softmax по сходствам якоря со всеми примерами батча, делённым на температуру $\tau$. Каждое обучение — 1500 шагов, около полутора секунд.

In [ ]:
def train_metric(loss_fn, steps=1500, extra_params=(), seed=0):
    """Обучает новый кодировщик на батчах m-per-class с заданной функцией потерь."""
    torch.manual_seed(seed)
    encoder = Encoder()
    # extra_params — обучаемые параметры самой потери (например, веса классов)
    optimizer = torch.optim.Adam([*encoder.parameters(), *extra_params], lr=1e-3)
    for xb, yb in class_batches(steps=steps, seed=seed):
        loss = loss_fn(encoder(xb), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return encoder

def info_nce(e, labels, tau=0.1):
    """InfoNCE с несколькими позитивами (supervised contrastive): softmax по всем примерам батча."""
    n = len(e)
    self_mask = torch.eye(n, dtype=torch.bool)
    logits = (e @ e.T / tau).masked_fill(self_mask, -float("inf"))   # сходства / τ, себя исключаем
    log_prob = logits - logits.logsumexp(dim=1, keepdim=True)        # log softmax по строке
    pos = (labels[:, None] == labels[None, :]) & ~self_mask
    # среднее −log p по всем позитивам якоря, затем по якорям
    return -(log_prob.masked_fill(~pos, 0).sum(dim=1) / pos.sum(dim=1)).mean()

head = nn.Linear(64, 70)                              # обычный классификатор на 70 чисел
t = time.perf_counter()
enc_clf = train_metric(lambda e, l: F.cross_entropy(head(e), l), extra_params=list(head.parameters()))
print(f"одно обучение: {time.perf_counter() - t:.1f} с")
enc_pair = train_metric(contrastive_loss)
enc_triplet = train_metric(triplet_loss)
enc_nce = train_metric(info_nce)
report("пиксели", lambda x: x)
report("классификатор", enc_clf)
report("парная", enc_pair)
report("тройная", enc_triplet)
report("InfoNCE", enc_nce)

* На знакомых числах все обученные модели примерно равны: 0.93–0.94.
* На **новых** числах классификатор даёт 0.815, а метрические потери — около 0.95: они учили общее правило «одинаковое — близко», которое переносится на новые классы.

### 1.6. Температура

InfoNCE при четырёх температурах. `cos_stats` считает на новых числах средний косинус между эмбеддингами одного числа («свои») и разных чисел («чужие»).

In [ ]:
def cos_stats(encoder):
    """Средний косинус между эмбеддингами одного числа и разных чисел (новые числа)."""
    with torch.no_grad():
        e = encoder(X_new)
    cos = e @ e.T
    same = (num_new[:, None] == num_new[None, :]) & ~torch.eye(len(e), dtype=torch.bool)
    diff = num_new[:, None] != num_new[None, :]
    return cos[same].mean().item(), cos[diff].mean().item()

for tau in [0.02, 0.1, 0.5, 1.0]:
    enc = train_metric(lambda e, l: info_nce(e, l, tau=tau))
    c_same, c_diff = cos_stats(enc)
    print(f"τ = {tau:4}: косинус своих {c_same:.2f}, чужих {c_diff:5.2f} |", end=" ")
    report("", enc)

* $\tau = 0.1$ — лучший компромисс: свои с косинусом 0.83, чужие около 0.
* При $\tau$ = 0.5 и 1 качество на новых числах падает.
* При $\tau = 0.02$ Recall@1 тот же, но чужие числа похожи друг на друга (косинус 0.65) — шаг к коллапсу.

### 1.7. Отступ в softmax: ArcFace, прокси и центры

**ArcFace**: классификатор по косинусу с масштабом $s = 30$, где угол до правильного класса при подсчёте потери увеличивается на отступ $m$. Сравним $m = 0$ и $m = 0.3$.

In [ ]:
class ArcFace(nn.Module):
    """Классификатор по косинусу с угловым отступом m для правильного класса."""
    def __init__(self, d_emb, n_classes, s=30.0, m=0.3):
        super().__init__()
        self.W = nn.Parameter(torch.randn(n_classes, d_emb) * 0.01)   # по вектору-представителю на класс
        self.s, self.m = s, m

    def forward(self, e, labels):
        cos = e @ F.normalize(self.W, dim=1).T        # cos θ_j до каждого класса
        theta = torch.acos(cos.clamp(-1 + 1e-6, 1 - 1e-6))
        target = torch.cos(theta + self.m)            # правильному классу угол увеличиваем на m
        is_true = F.one_hot(labels, cos.shape[1]).bool()
        logits = self.s * torch.where(is_true, target, cos)
        return F.cross_entropy(logits, labels)

for m in [0.0, 0.3]:
    arc = ArcFace(64, 70, m=m)
    enc = train_metric(arc, extra_params=list(arc.parameters()))
    report(f"ArcFace m={m}", enc)

**Что получилось.** Отступ поднял Recall@1 на новых числах с 0.910 до 0.940 — почти до уровня InfoNCE.

### 1.8. Как проверить пространство эмбеддингов

Диагностика пространства на новых числах: средний косинус случайных пар, средний радиус класса, расстояние между ближайшими центрами классов и виды троек. Сравним необученный кодировщик, InfoNCE с $\tau = 0.1$ и с $\tau = 0.02$.

In [ ]:
def diagnose(name, encoder):
    """Средний косинус случайных пар, внутриклассовый и межклассовый разброс, виды троек."""
    with torch.no_grad():
        e = encoder(X_new)
    centers = torch.stack([e[num_new == c].mean(0) for c in new])   # центры 30 новых чисел
    intra = torch.stack([(e[num_new == c] - centers[i]).norm(dim=1).mean() for i, c in enumerate(new)]).mean()
    inter = torch.pdist(centers).min()                # расстояние между двумя ближайшими центрами
    g = torch.Generator().manual_seed(0)
    i, j = torch.randint(len(e), (2, 2000), generator=g)
    print(f"{name:13s} косинус случайных пар {(e[i] * e[j]).sum(1).mean():.2f} | "
          f"радиус класса {intra:.2f} | ближайшие центры {inter:.2f} | {triplet_types(e[:200], num_new[:200])}")

torch.manual_seed(0)
diagnose("необученный", Encoder())
diagnose("InfoNCE", enc_nce)
diagnose("τ = 0.02", train_metric(lambda e, l: info_nce(e, l, tau=0.02)))

* Необученный: эмбеддинги в узком конусе (косинус 0.59), классы перекрываются.
* InfoNCE: косинус 0.05, классы компактны и разнесены.
* $\tau = 0.02$: классы компактны, но всё пространство сжалось — косинус 0.66.

## Глава 2. Поиск похожих объектов

### 2.2. Точный поиск

Галерея из 100 тысяч эмбеддингов и 1000 запросов. Точный поиск — одно умножение матриц `Q @ G.T` и `topk`; его результат `true_top` — эталон для приближённых методов. `search_quality` считает recall@10 (доля настоящих 10 ближайших среди найденных) и precision@10 (доля найденных с тем же числом, что у запроса).

In [ ]:
def encode_numbers(n, seed):
    """n случайных двузначных чисел из проверочных картинок → эмбеддинги кодировщика enc_nce."""
    X, nums = make_numbers(Xd_test, yd_test, torch.randint(100, (n,), generator=torch.Generator().manual_seed(seed)), seed)
    with torch.no_grad():
        return enc_nce((X - mu) / sd), nums

G, num_G = encode_numbers(100_000, seed=10)           # галерея: 100 тысяч эмбеддингов
Q, num_Q = encode_numbers(1_000, seed=11)             # 1000 запросов
print("галерея:", tuple(G.shape), f"| {G.numel() * 4 / 2**20:.0f} МБ в float32")

t = time.perf_counter()
true_top = (Q @ G.T).topk(10, dim=1).indices          # точный поиск: сходство со всей галереей
print(f"точный поиск: {1000 * (time.perf_counter() - t):.0f} мс на 1000 запросов")

def search_quality(found):
    """recall@10 — доля настоящих 10 ближайших среди найденных; precision@10 — доля найденных того же числа."""
    rec = sum(len(set(f.tolist()) & set(t.tolist())) for f, t in zip(found, true_top)) / (10 * len(found))
    prec = sum((num_G[f] == num_Q[i]).float().mean().item() for i, f in enumerate(found)) / len(found)
    return f"recall@10 {rec:.3f}, precision@10 {prec:.3f}"

def top10_among(i, candidates):
    """Точное ранжирование запроса i среди кандидатов: 10 самых похожих."""
    sims = G[candidates] @ Q[i]
    return candidates[sims.topk(min(10, len(candidates))).indices]

print("точный поиск:", search_quality(list(true_top)))

**Что получилось.** Галерея — 24 МБ, 1000 запросов — около десятой доли секунды. 96% найденного — то же число; остальные 4% — ошибки эмбеддингов, а не поиска.

### 2.3. LSH, IVF и HNSW: смотреть не на всех

**LSH**: код объекта — с какой стороны от случайных гиперплоскостей он лежит; кандидаты — объекты с тем же кодом хотя бы в одной таблице. Скорость сравниваем по доле просмотренной галереи.

In [ ]:
def lsh_search(n_bits, n_tables, seed=0):
    """Код объекта — с какой стороны от n_bits случайных гиперплоскостей он лежит; кандидаты — совпавшие коды."""
    torch.manual_seed(seed)
    planes = torch.randn(n_tables, 64, n_bits)        # своя пачка гиперплоскостей на каждую таблицу
    powers = 2 ** torch.arange(n_bits)                # биты → целое число (номер корзины)
    codes_G = ((torch.einsum("nd,tdb->tnb", G, planes) > 0).long() * powers).sum(-1)   # (таблицы, N)
    codes_Q = ((torch.einsum("nd,tdb->tnb", Q, planes) > 0).long() * powers).sum(-1)
    found, scanned = [], 0
    for i in range(len(Q)):
        # кандидат — объект, попавший в ту же корзину хотя бы в одной таблице
        candidates = (codes_G == codes_Q[:, i:i + 1]).any(dim=0).nonzero().flatten()
        scanned += len(candidates)
        found.append(top10_among(i, candidates))
    print(f"LSH: {n_bits:2d} бит, {n_tables} табл. | просмотрено {100 * scanned / len(Q) / len(G):4.1f}% | {search_quality(found)}")

for n_bits, n_tables in [(8, 1), (8, 4), (12, 4)]:
    lsh_search(n_bits, n_tables)

**Что получилось.** Одна таблица из 8 битов просматривает 1% галереи, но находит лишь 59% соседей. Четыре таблицы — 95% при 4% просмотра.

**IVF**: галерея разбита методом k-средних на 256 кластеров; запрос ищут только в `nprobe` ближайших кластерах. Построение индекса занимает несколько секунд.

In [ ]:
from sklearn.cluster import KMeans

t = time.perf_counter()
kmeans = KMeans(n_clusters=256, n_init=1, random_state=0).fit(G.numpy())   # 256 центроидов
centroids = torch.tensor(kmeans.cluster_centers_, dtype=torch.float32)
lists = [torch.nonzero(torch.tensor(kmeans.labels_) == c).flatten() for c in range(256)]   # «списки» кластеров
print(f"построение индекса: {time.perf_counter() - t:.1f} с")

for nprobe in [1, 4, 16]:
    nearest_clusters = (Q @ centroids.T).topk(nprobe, dim=1).indices   # nprobe ближайших центроидов
    found, scanned = [], 0
    for i in range(len(Q)):
        candidates = torch.cat([lists[c] for c in nearest_clusters[i]])
        scanned += len(candidates)
        found.append(top10_among(i, candidates))
    print(f"IVF: nprobe = {nprobe:2d} | просмотрено {100 * scanned / len(Q) / len(G):4.1f}% | {search_quality(found)}")

**Что получилось.** При `nprobe = 4` найдено 99% соседей при просмотре 1.5% галереи — в 65 раз меньше сравнений, чем у точного поиска.

### 2.4. PQ: сжать векторы, а потом уточнить

**PQ**: каждый вектор режется на 8 кусков по 8 чисел, и каждый кусок заменяется номером ближайшего из 256 центроидов — 8 байт вместо 256. Сравним поиск прямо по сжатым векторам и с точным переранжированием 100 кандидатов.

In [ ]:
M, K = 8, 256                                         # 8 кусков по 8 чисел, 256 центроидов на кусок
sub = 64 // M
codebooks, codes = [], []
for j in range(M):
    km = KMeans(n_clusters=K, n_init=1, random_state=0).fit(G[:, j * sub:(j + 1) * sub].numpy())
    codebooks.append(torch.tensor(km.cluster_centers_, dtype=torch.float32))
    codes.append(torch.tensor(km.labels_))
codes = torch.stack(codes, dim=1)                     # (N, 8): номера центроидов — по байту на кусок
print(f"память на вектор: {M} байт вместо {64 * 4} | вся галерея {len(G) * M / 2**20:.1f} МБ")

# таблицы сходств: скалярное произведение куска запроса с каждым центроидом этого куска
tables = torch.stack([Q[:, j * sub:(j + 1) * sub] @ codebooks[j].T for j in range(M)], dim=1)   # (запросы, M, K)
approx = sum(tables[:, j, codes[:, j]] for j in range(M))          # приближённое сходство со всей галереей
print("PQ:", search_quality(list(approx.topk(10, dim=1).indices)))
candidates = approx.topk(100, dim=1).indices          # первый этап: 100 кандидатов по PQ
print("PQ, 100 кандидатов + точное переранжирование:",
      search_quality([top10_among(i, candidates[i]) for i in range(len(Q))]))

* Память — 8 байт на вектор, в 32 раза меньше.
* Без уточнения recall@10 всего 0.09, но precision@10 — 0.91: ошибки сжатия переставляют почти одинаково близкие картинки одного числа.
* С переранжированием precision@10 — 0.95, почти как у точного поиска.

### 2.5. Метрики поиска

DCG и nDCG для трёх выдач с одинаковым набором оценок релевантности 3, 2, 1, 0, 0 в разном порядке.

In [ ]:
def dcg(rel):
    """DCG: релевантность позиции i (с единицы) делится на log2(i + 1)."""
    rel = torch.tensor(rel, dtype=torch.float32)
    return (rel / torch.log2(torch.arange(2, len(rel) + 2).float())).sum().item()

def ndcg(rel):
    return dcg(rel) / dcg(sorted(rel, reverse=True))  # делим на DCG идеального порядка

for rel in [[3, 2, 1, 0, 0], [0, 0, 1, 2, 3], [3, 0, 0, 2, 1]]:   # релевантность выдачи по позициям
    print(f"выдача {rel}: DCG@5 {dcg(rel):.2f}, nDCG@5 {ndcg(rel):.2f}")

**Что получилось.** Идеальный порядок — 1.00, перевёрнутый — 0.53, лучший на первом месте при перемешанных остальных — 0.89: ошибки в хвосте выдачи стоят меньше.

## Глава 3. Перенос обучения

### 3.2. Заморозить или доучить всё

Перенос обучения: кодировщик `enc_nce` — «предобученная модель», новая задача — классификация 30 новых чисел по $K = 5$ картинок на число. `finetune` поддерживает все режимы главы; здесь сравним обучение с нуля, только голову (кодировщик заморожен) и полное дообучение.

In [ ]:
# новая задача — классифицировать 30 новых чисел; размеченных картинок мало: K на число
idx_new = torch.searchsorted(new.sort().values, num_new)   # номер класса 0–29 у проверочных картинок

def target_data(K, seed):
    """K размеченных картинок каждого из 30 новых чисел (из обучающих картинок цифр)."""
    X, nums = make_numbers(Xd_train, yd_train, new.repeat_interleave(K), seed=100 + seed)
    return (X - mu) / sd, torch.searchsorted(new.sort().values, nums)

class Classifier(nn.Module):
    """Кодировщик + новая голова на 30 классов."""
    def __init__(self, encoder):
        super().__init__()
        self.encoder, self.head = encoder, nn.Linear(64, 30)

    def forward(self, x):
        return self.head(10 * self.encoder(x))        # ×10: у эмбеддинга длина 1, логитам нужен размах

def finetune(mode, K=5, seed=0, epochs=100, l2sp=0.0):
    """Дообучает предобученный enc_nce (или новый кодировщик) на K примерах каждого нового числа."""
    X, y = target_data(K, seed)
    torch.manual_seed(seed)
    encoder = Encoder() if mode == "с нуля" else copy.deepcopy(enc_nce)
    model = Classifier(encoder)
    start = [p.detach().clone() for p in encoder.parameters()]   # предобученные веса — для L2-SP
    net = encoder.net                                 # net[0], net[2], net[4] — три линейных слоя
    if mode == "голова":                              # feature extraction: кодировщик заморожен
        encoder.requires_grad_(False)
        groups = [{"params": model.head.parameters(), "lr": 1e-2}]
    elif mode == "частично":                          # размораживаем только последний слой
        net[:4].requires_grad_(False)
        groups = [{"params": model.head.parameters(), "lr": 1e-2},
                  {"params": net[4].parameters(), "lr": 1e-3}]
    elif mode == "разные lr":                         # дискриминативные скорости: ниже слой — меньше lr
        groups = [{"params": model.head.parameters(), "lr": 1e-2},
                  {"params": net[4].parameters(), "lr": 1e-3},
                  {"params": net[2].parameters(), "lr": 3e-4},
                  {"params": net[0].parameters(), "lr": 1e-4}]
    else:                                             # «с нуля» и «всё»: одна скорость для всех весов
        groups = [{"params": model.parameters(), "lr": 1e-3}]
    optimizer = torch.optim.Adam(groups)
    for epoch in range(epochs):
        order = torch.randperm(len(X))
        for i in range(0, len(X), 32):
            b = order[i:i + 32]
            loss = F.cross_entropy(model(X[b]), y[b])
            if l2sp:                                  # штраф за уход от предобученных весов
                loss = loss + l2sp / 2 * sum(((p - p0) ** 2).sum() for p, p0 in zip(encoder.parameters(), start))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    with torch.no_grad():
        return (model(X_new).argmax(dim=1) == idx_new).float().mean().item()

def compare(modes, K=5, **kw):
    for mode in modes:
        accs = [finetune(mode, K=K, seed=s, **kw) for s in range(3)]   # три разных набора K примеров
        print(f"{mode:9s} K = {K:2d}: точность {sum(accs) / 3:.3f}  (запуски: {', '.join(f'{a:.3f}' for a in accs)})")

compare(["с нуля", "голова", "всё"])

* С нуля — 0.64: 150 примеров мало.
* Только голова — 0.92: предобученный кодировщик уже различает новые числа.
* Полное дообучение — 0.82, хуже одной головы: большие градиенты случайной головы портят признаки (катастрофическое забывание).

### 3.3. Частичное дообучение и разные скорости обучения

Режимы «частично» (учится только последний слой кодировщика и голова) и «разные lr» (скорость убывает от головы вниз).

In [ ]:
compare(["частично", "разные lr"])

**Что получилось.** Оба варианта — около 0.92–0.93, намного лучше полного дообучения с одной скоростью.

### 3.4. Регуляризация к исходным весам: L2-SP

Полное дообучение со штрафом **L2-SP** за уход весов от предобученных при трёх силах штрафа $\lambda$.

In [ ]:
for lam in [0.1, 1.0, 10.0]:
    accs = [finetune("всё", K=5, seed=s, l2sp=lam) for s in range(3)]
    print(f"всё + L2-SP, λ = {lam:4}: точность {sum(accs) / 3:.3f}")

**Что получилось.** При $\lambda = 10$ полное дообучение даёт 0.93 вместо 0.82.

### 3.5. Сколько нужно размеченных данных

Пять режимов при 2, 5 и 20 примерах на число, каждый — среднее трёх запусков. Самая долгая ячейка главы, около 20 секунд.

In [ ]:
for K in [2, 5, 20]:
    row = {mode: sum(finetune(mode, K=K, seed=s) for s in range(3)) / 3
           for mode in ["с нуля", "голова", "частично", "всё", "разные lr"]}
    print(f"K = {K:2d}: " + " | ".join(f"{m} {a:.3f}" for m, a in row.items()))

**Что получилось.** С ростом $K$ полное дообучение догоняет остальных, а при $K = 20$ лучший вариант — разные скорости обучения (0.958).

### 3.6. Методы, экономные по параметрам: адаптеры и LoRA

**LoRA**: веса кодировщика заморожены, а ко второму и третьему слою добавлена обучаемая поправка $BA$ ранга 4.

In [ ]:
class LoRALinear(nn.Module):
    """Замороженный линейный слой + обучаемая добавка низкого ранга: W x + (B A) x · alpha / r."""
    def __init__(self, base, r=4, alpha=8):
        super().__init__()
        self.base = base.requires_grad_(False)        # исходные веса не меняются
        self.A = nn.Parameter(torch.randn(r, base.in_features) / base.in_features ** 0.5)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))   # B = 0: в начале добавка нулевая
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale

lora_enc = copy.deepcopy(enc_nce)
lora_enc.net[2], lora_enc.net[4] = LoRALinear(lora_enc.net[2]), LoRALinear(lora_enc.net[4])
lora_enc.net[0].requires_grad_(False)                 # первый слой просто заморожен
with torch.no_grad():
    same = torch.allclose(lora_enc(X_new), enc_nce(X_new))
trainable = sum(p.numel() for p in lora_enc.parameters() if p.requires_grad)
total = sum(p.numel() for p in enc_nce.parameters())
print(f"до обучения выход не изменился: {same} | обучаемых {trainable} из {total} ({100 * trainable / total:.1f}%)")

def finetune_lora(K=5, seed=0, epochs=100):
    X, y = target_data(K, seed)
    torch.manual_seed(seed)
    model = Classifier(copy.deepcopy(lora_enc))
    optimizer = torch.optim.Adam([{"params": model.head.parameters(), "lr": 1e-2},
                                  {"params": [p for n, p in model.encoder.named_parameters() if n.endswith((".A", ".B"))],
                                   "lr": 1e-3}])
    for epoch in range(epochs):
        order = torch.randperm(len(X))
        for i in range(0, len(X), 32):
            b = order[i:i + 32]
            loss = F.cross_entropy(model(X[b]), y[b])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    with torch.no_grad():
        return (model(X_new).argmax(dim=1) == idx_new).float().mean().item()

accs = [finetune_lora(seed=s) for s in range(3)]
print(f"LoRA      K =  5: точность {sum(accs) / 3:.3f}  (запуски: {', '.join(f'{a:.3f}' for a in accs)})")

**Что получилось.** До обучения выход не изменился ($BA = 0$, потому что `B` начинается с нулей). Обучается 2.9% параметров кодировщика, а точность 0.92 — на уровне лучших режимов.

## Глава 4. Адаптация к новому домену

### 4.1. Сдвиг домена

Адаптация домена. Снова одиночные цифры 8×8: исходный домен — обычные, целевой — «другой сканер» (размытие, ниже контраст, светлее фон, шум). Целевые данные без меток — `Xt_unlab`, для итоговой проверки — `Xt_test`. Обучаем сеть с BatchNorm только на исходном домене и, для сравнения, на целевом с метками.

In [ ]:
# снова одиночные цифры 8×8: исходный домен — обычные, целевой — размытые и бледные
Xs, Xt, ys, yt = train_test_split(X_digits.view(-1, 1, 8, 8), y_digits, test_size=0.5, random_state=0, stratify=y_digits)
blur = torch.tensor([[1., 2., 1.], [2., 4., 2.], [1., 2., 1.]]).view(1, 1, 3, 3) / 16   # ядро размытия

def other_scanner(img):
    """Размытие, контраст 0.8, фон +0.1 и шум — так «видит» цифры другой сканер."""
    g = torch.Generator().manual_seed(0)
    blurred = F.conv2d(F.pad(img, (1, 1, 1, 1), mode="replicate"), blur)
    return 0.8 * blurred + 0.1 + 0.05 * torch.randn(blurred.shape, generator=g)

Xt = other_scanner(Xt)
# целевой домен делим пополам: без меток — для адаптации, с метками — только для итоговой проверки
Xt_unlab, Xt_test, yt_unlab_hidden, yt_test = train_test_split(Xt, yt, test_size=0.5, random_state=0, stratify=yt)
Xs_train, Xs_test, ys_train, ys_test = train_test_split(Xs, ys, test_size=0.2, random_state=0, stratify=ys)
m_s, s_s = Xs_train.view(len(Xs_train), -1).mean(0), Xs_train.view(len(Xs_train), -1).std(0) + 0.1
prep = lambda t: (t.view(len(t), -1) - m_s) / s_s     # стандартизация по исходному домену
Xs_train, Xs_test, Xt_unlab, Xt_test = prep(Xs_train), prep(Xs_test), prep(Xt_unlab), prep(Xt_test)
print("исходный домен:", len(Xs_train), "с метками | целевой:", len(Xt_unlab), "без меток +", len(Xt_test), "для проверки")

def make_net():
    """Признаки (два слоя с BatchNorm) и классификатор."""
    features = nn.Sequential(nn.Linear(64, 128), nn.BatchNorm1d(128), nn.ReLU(),
                             nn.Linear(128, 128), nn.BatchNorm1d(128), nn.ReLU())
    return features, nn.Linear(128, 10)

def accuracy(features, head, X, y):
    features.eval(), head.eval()
    with torch.no_grad():
        return (head(features(X)).argmax(dim=1) == y).float().mean().item()

def train_da(seed, extra_loss=None, X=None, y=None, epochs=40, extra_module=None):
    """Обучение на исходном домене; extra_loss(features, head, xs, ys, xt, progress) добавляет адаптацию."""
    X = Xs_train if X is None else X
    y = ys_train if y is None else y
    torch.manual_seed(seed)
    features, head = make_net()
    params = [*features.parameters(), *head.parameters()] + (list(extra_module.parameters()) if extra_module else [])
    optimizer = torch.optim.Adam(params, lr=1e-3)
    for epoch in range(epochs):
        order = torch.randperm(len(X))
        for i in range(0, len(X) - 1, 64):
            b = order[i:i + 64]
            features.train(), head.train()
            if extra_loss is None:
                loss = F.cross_entropy(head(features(X[b])), y[b])
            else:
                xt = Xt_unlab[torch.randint(len(Xt_unlab), (64,))]   # случайный батч целевых без меток
                loss = extra_loss(features, head, X[b], y[b], xt, epoch / epochs)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return features, head

def over_seeds(name, fn):
    """Запускает fn(seed) → точность на целевом домене для трёх сидов."""
    accs = [fn(s) for s in range(3)]
    print(f"{name:22s}: целевой домен {sum(accs) / 3:.3f}  (запуски: {', '.join(f'{a:.3f}' for a in accs)})")

source_models = [train_da(s) for s in range(3)]       # обычное обучение только на исходном домене
f0, h0 = source_models[0]
print(f"исходный домен: {accuracy(f0, h0, Xs_test, ys_test):.3f} | целевой домен: {accuracy(f0, h0, Xt_test, yt_test):.3f}")
over_seeds("только исходный домен", lambda s: accuracy(*source_models[s], Xt_test, yt_test))
over_seeds("с метками целевого", lambda s: accuracy(*train_da(s, X=Xt_unlab, y=yt_unlab_hidden), Xt_test, yt_test))

**Что получилось.** На исходном домене — 0.97, на целевом — в среднем 0.54. С метками целевого домена — 0.92: задача решаема, модель просто училась на других данных.

### 4.3. AdaBN: пересчитать статистики нормализации

**AdaBN**: пересчитываем статистики всех слоёв BatchNorm по целевым данным без меток — без обучения, одним прямым проходом.

In [ ]:
def adabn(features):
    """Копия признаков, у которой статистики BatchNorm пересчитаны по целевому домену."""
    f = copy.deepcopy(features)
    for m in f.modules():
        if isinstance(m, nn.BatchNorm1d):
            m.reset_running_stats()
            m.momentum = None                         # None — простое среднее по всем батчам
    f.train()                                         # в режиме train BatchNorm копит статистику
    with torch.no_grad():
        f(Xt_unlab)                                   # один проход по целевым данным без меток
    return f

over_seeds("AdaBN", lambda s: accuracy(adabn(source_models[s][0]), source_models[s][1], Xt_test, yt_test))

**Что получилось.** 0.54 → 0.85 без единой метки и без обучения: наш сдвиг меняет в основном средние и разбросы признаков.

### 4.4. Псевдоразметка

**Псевдометки**: модель после AdaBN размечает целевые данные; уверенные предсказания (вероятность выше 0.9) добавляем к обучающим и обучаем заново.

In [ ]:
def self_training(seed, threshold=0.9):
    features, head = source_models[seed]
    features = adabn(features)
    features.eval(), head.eval()
    with torch.no_grad():
        prob = F.softmax(head(features(Xt_unlab)), dim=1)
    conf, pseudo = prob.max(dim=1)                    # уверенность и предсказанный класс
    keep = conf > threshold                           # берём только уверенные предсказания
    if seed == 0:
        print(f"уверенных: {keep.float().mean():.0%} целевых примеров, из них верно размечено "
              f"{(pseudo[keep] == yt_unlab_hidden[keep]).float().mean():.0%}")
    # заново обучаем на исходных + псевдоразмеченных целевых
    f, h = train_da(seed, X=torch.cat([Xs_train, Xt_unlab[keep]]), y=torch.cat([ys_train, pseudo[keep]]))
    return accuracy(adabn(f), h, Xt_test, yt_test)

over_seeds("AdaBN + псевдометки", self_training)

**Что получилось.** Уверенных — 54% целевых примеров, и 96% их псевдометок верны. Точность выросла до 0.88.

### 4.5. Выравнивание признаков: CORAL и MMD

**CORAL**: к кросс-энтропии добавляется штраф за разницу ковариационных матриц признаков исходного и целевого батчей.

In [ ]:
def coral_loss(features, head, xs, ys, xt, progress, lam=1.0):
    fs, ft = features(xs), features(xt)
    d = fs.shape[1]
    # разница ковариационных матриц признаков двух доменов (квадрат нормы Фробениуса)
    coral = ((torch.cov(fs.T) - torch.cov(ft.T)) ** 2).sum() / (4 * d * d)
    return F.cross_entropy(head(fs), ys) + lam * coral

over_seeds("CORAL", lambda s: accuracy(*train_da(s, coral_loss), Xt_test, yt_test))

**Что получилось.** CORAL поднял точность до 0.70 — лучше, чем без адаптации, но хуже AdaBN.

### 4.6. Состязательная адаптация: DANN

**DANN**: классификатор доменов учится отличать исходный домен от целевого, а слой разворота градиента `GradReverse` заставляет признаки быть неразличимыми. Сначала проверим сам слой разворота.

In [ ]:
class GradReverse(torch.autograd.Function):
    """Прямой проход — тождественный, обратный — градиент умножается на −λ."""
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad):
        return -ctx.lam * grad, None                  # None — градиент по lam не нужен

x = torch.ones(3, requires_grad=True)
(GradReverse.apply(x, 0.5) * 2).sum().backward()
print("градиент через разворот:", x.grad.tolist())    # без разворота было бы [2, 2, 2]

discriminator = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))   # «исходный или целевой?»

def dann_loss(features, head, xs, ys, xt, progress):
    lam = 0.3 * min(1.0, 2 * progress)                # λ плавно растёт от 0 до 0.3
    fs, ft = features(xs), features(xt)
    domain_logits = discriminator(GradReverse.apply(torch.cat([fs, ft]), lam)).squeeze(1)
    domain = torch.cat([torch.zeros(len(fs)), torch.ones(len(ft))])   # 0 — исходный, 1 — целевой
    return F.cross_entropy(head(fs), ys) + F.binary_cross_entropy_with_logits(domain_logits, domain)

def run_dann(seed):
    global discriminator
    torch.manual_seed(100 + seed)
    discriminator = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))
    return accuracy(*train_da(seed, dann_loss, extra_module=discriminator), Xt_test, yt_test)

over_seeds("DANN", run_dann)

**Что получилось.** Градиент через разворот равен $-1$ вместо 2. DANN дал 0.67 — лучше, чем без адаптации, но хуже AdaBN: состязательное обучение капризно.

## Глава 5. Многозадачное обучение

### 5.1. Одна сеть — несколько задач

Многозадачная сеть: общее тело и три головы — цифра десятков, цифра единиц и значение числа (регрессия). Посмотрим на потери и нормы градиентов задач по общему телу у необученной сети.

In [ ]:
X_mt, num_mt = make_numbers(Xd_train, yd_train, torch.arange(100).repeat_interleave(10), seed=20)
X_mt_test, num_mt_test = make_numbers(Xd_test, yd_test, torch.arange(100).repeat_interleave(10), seed=21)
X_mt, X_mt_test = (X_mt - mu) / sd, (X_mt_test - mu) / sd

class MultiTask(nn.Module):
    """Общее тело и три головы: цифра десятков, цифра единиц, значение числа."""
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(128, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU())
        self.tens, self.ones, self.value = nn.Linear(256, 10), nn.Linear(256, 10), nn.Linear(256, 1)

    def forward(self, x):
        h = self.body(x)                              # общие признаки для всех задач
        return self.tens(h), self.ones(h), self.value(h).squeeze(1)

def task_losses(model, x, num):
    tens, ones, value = model(x)
    return (F.cross_entropy(tens, num // 10),         # задача 1: классификация, 10 классов
            F.cross_entropy(ones, num % 10),          # задача 2: классификация, 10 классов
            F.mse_loss(value, num.float()))           # задача 3: регрессия, число от 0 до 99

def shared_grads(model, losses):
    """Градиент каждой потери по общим весам тела, вытянутый в один вектор."""
    return [torch.cat([g.flatten() for g in torch.autograd.grad(l, list(model.body.parameters()), retain_graph=True)])
            for l in losses]

torch.manual_seed(0)
model = MultiTask()
b = torch.randperm(len(X_mt))[:256]
losses = task_losses(model, X_mt[b], num_mt[b])
print("потери до обучения:", [round(l.item(), 2) for l in losses])
print("нормы градиентов по общему телу:", [round(g.norm().item(), 3) for g in shared_grads(model, losses)])

**Что получилось.** Потери классификаций около $\ln 10 \approx 2.3$, потеря регрессии — больше трёх тысяч, а её градиент по телу почти в 400 раз длиннее. Простая сумма потерь учила бы почти только регрессию.

### 5.2. Сравнение с отдельными моделями

Сравним однозадачные модели, простую сумму потерь и веса 1, 1, 0.001.

In [ ]:
def train_mtl(weights=(1, 1, 1), uncertainty=False, epochs=60, seed=0):
    """Обучает MultiTask; weights — веса потерь, uncertainty=True — веса через обучаемые σ."""
    torch.manual_seed(seed)
    model = MultiTask()
    log_sigma = torch.zeros(3, requires_grad=True)    # log σ_t для каждой задачи
    optimizer = torch.optim.Adam([{"params": model.parameters(), "lr": 1e-3},
                                  {"params": [log_sigma], "lr": 5e-2}])
    for epoch in range(epochs):
        order = torch.randperm(len(X_mt))
        for i in range(0, len(X_mt), 64):
            b = order[i:i + 64]
            L = task_losses(model, X_mt[b], num_mt[b])
            if uncertainty:                           # Σ L_t / (2σ_t²) + log σ_t
                loss = sum(0.5 * torch.exp(-2 * log_sigma[t]) * L[t] + log_sigma[t] for t in range(3))
            else:
                loss = sum(w * l for w, l in zip(weights, L) if w)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        tens, ones, value = model(X_mt_test)
    res = ((tens.argmax(1) == num_mt_test // 10).float().mean().item(),
           (ones.argmax(1) == num_mt_test % 10).float().mean().item(),
           (value - num_mt_test.float()).abs().mean().item())
    return model, res, log_sigma.exp().detach()

def show(name, res):
    print(f"{name:26s}: десятки {res[0]:.3f} | единицы {res[1]:.3f} | ошибка значения {res[2]:5.2f}")

for name, w in [("только десятки", (1, 0, 0)), ("только единицы", (0, 1, 0)), ("только значение", (0, 0, 1)),
                ("сумма потерь", (1, 1, 1)), ("веса 1, 1, 0.001", (1, 1, 0.001))]:
    show(name, train_mtl(weights=w)[1])

* Простая сумма потерь: точность классификаций упала до 0.83 и 0.79.
* Веса 1, 1, 0.001 вернули классификации к уровню отдельных моделей, а ошибка значения даже уменьшилась до 10.6 против 11.4 у отдельной модели.

### 5.3. Балансировка потерь

Взвешивание по неопределённости: веса задач $1/(2\sigma_t^2)$ становятся обучаемыми.

In [ ]:
model_u, res, sigma = train_mtl(uncertainty=True)
show("неопределённость", res)
print("выученные σ:", [round(s, 3) for s in sigma.tolist()],
      "| веса 1/(2σ²):", [round(w, 4) for w in (0.5 / sigma ** 2).tolist()])

**Что получилось.** Веса подобрались сами: у регрессии около 0.011, у классификаций 8.5 и 42 — тот же баланс, что мы нашли вручную.

### 5.4. Конфликты градиентов

**PCGrad** на двух векторах на плоскости, а затем косинусы между градиентами трёх задач по общему телу.

In [ ]:
def pcgrad(grads):
    """PCGrad: если градиенты двух задач конфликтуют (cos < 0), убираем из одного проекцию на другой."""
    projected = [g.clone() for g in grads]
    for i, g_i in enumerate(projected):
        for j, g_j in enumerate(grads):
            dot = g_i @ g_j
            if i != j and dot < 0:                    # конфликт: тянут в разные стороны
                g_i -= dot / (g_j @ g_j) * g_j        # вычитаем проекцию на g_j
    return sum(projected)

g1, g2 = torch.tensor([1.0, 1.0]), torch.tensor([-1.0, 0.5])
print("cos(g1, g2) =", round(F.cosine_similarity(g1, g2, dim=0).item(), 2),
      "| сумма:", (g1 + g2).tolist(), "| PCGrad:", [round(v, 2) for v in pcgrad([g1, g2]).tolist()])

model_sum = train_mtl(weights=(1, 1, 1))[0]           # модель с простой суммой потерь
b = torch.randperm(len(X_mt), generator=torch.Generator().manual_seed(1))[:256]
grads = shared_grads(model_sum, task_losses(model_sum, X_mt[b], num_mt[b]))
names = ["десятки", "единицы", "значение"]
for i, j in [(0, 1), (0, 2), (1, 2)]:
    print(f"cos({names[i]}, {names[j]}) = {F.cosine_similarity(grads[i], grads[j], dim=0):.2f}")

**Что получилось.** Наши задачи почти не конфликтуют: косинусы от 0 до 0.18. Проблема простой суммы — только в масштабе потерь, и PCGrad здесь не нужен.